In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

#1
class SUIMDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.split = split
        self.transform = transform

        self.image_dir = os.path.join(root_dir, split, 'images')
        self.mask_dir = os.path.join(root_dir, split, 'masks')

        self.image_filenames = sorted([f for f in os.listdir(self.image_dir) if f.endswith(('.jpg', '.png'))])
        self.mask_filenames = sorted([f for f in os.listdir(self.mask_dir) if f.endswith(('.jpg', '.png'))])

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        path = os.path.join(self.image_dir, self.image_filenames[idx])
        path = os.path.join(self.mask_dir, self.mask_filenames[idx])

        image = Image.open(path).convert("RGB")
        mask = Image.open(path).convert("L")

        image = image.resize((256, 256), Image.BILINEAR)
        mask = mask.resize((256, 256), Image.NEAREST)

        if self.transform:
            image = self.transform(image)

        mask = torch.from_numpy(np.array(mask)).long()

        return image, mask

#2
img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = SUIMDataset(root_dir=path, split='train', transform=img_transform)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

print(f"Dataset initialized with {len(train_dataset)} images.")

#3
def display_samples(dataset, num_samples=3):
    plt.figure(figsize=(12, num_samples * 4))

    for i in range(num_samples):
        img, mask = dataset[i]

        img_np = img.permute(1, 2, 0).numpy()
        img_np = img_np * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
        img_np = np.clip(img_np, 0, 1)

        plt.subplot(num_samples, 2, i*2 + 1)
        plt.imshow(img_np)
        plt.title(f"Underwater Image {i+1}")
        plt.axis('off')

        plt.subplot(num_samples, 2, i*2 + 2)
        plt.imshow(mask.numpy(), cmap='tab10')
        plt.title(f"Ground Truth Mask {i+1}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

display_samples(train_dataset)

In [ ]:
# TO DO
!pip install segmentation_models_pytorch

import segmentation_models_pytorch as smp

def build_unet_model(num_classes=8):
    """
    Builds a U-Net model with an EfficientNet-B1 encoder.
    """
    model = smp.Unet(
        encoder_name="efficientnet-b1",         #pretrained backbone
        encoder_weights="imagenet",             #use weights from ImageNet training
        in_channels=3,                          #RGB input
        classes=num_classes,                    #8 classes for SUIM
        activation=None                         #output raw logits
    )

    return model

model = build_unet_model(num_classes=8)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model successfully loaded with EfficientNet-B1 encoder.")
print(f"Output channels: {model.segmentation_head[0].out_channels} (One per class)")

In [ ]:
# TO DO
import torch
import torch.nn as nn
import torch.optim as optim

#1
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

#2
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct_pixels = 0
    total_pixels = 0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        #forward pass
        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        correct_pixels += (preds == masks).sum().item()
        total_pixels += masks.nelement()

    train_loss = running_loss / len(loader)
    train_acc = (correct_pixels / total_pixels) * 100
    return train_loss, train_acc

#3
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_pixels = 0
    total_pixels = 0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            running_loss += loss.item()

            #calculate pixel accuracy
            preds = torch.argmax(outputs, dim=1)
            correct_pixels += (preds == masks).sum().item()
            total_pixels += masks.nelement()

    val_loss = running_loss / len(loader)
    val_acc = (correct_pixels / total_pixels) * 100
    return val_loss, val_acc



In [ ]:
# TO DO
import torch.optim as optim
import matplotlib.pyplot as plt

#1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

train_losses = []
val_losses = []
num_epochs = 15


#2
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)

    val_loss, val_acc = validate_one_epoch(model, train_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"| Train Loss: {train_loss:.4f} "
          f"| Val Loss: {val_loss:.4f} "
          f"| Val Acc: {val_acc:.2f}%")

#3
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss', marker='o')
plt.plot(range(1, num_epochs + 1), val_losses, label='Validation Loss', marker='s')
plt.title('Multi-Class Segmentation: Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Cross Entropy Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# TO DO
import torch
import numpy as np
import matplotlib.pyplot as plt

def visualize_predictions(model, dataset, device, num_samples=3):
    model.eval()

    fig, axes = plt.subplots(num_samples, 3, figsize=(15, num_samples * 5))

    indices = np.random.choice(len(dataset), num_samples, replace=False)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            image, mask = dataset[idx]

            input_tensor = image.unsqueeze(0).to(device)

            #forward pass
            output = model(input_tensor)

            prediction = torch.argmax(output.squeeze(), dim=0).cpu().numpy()

            #1
            img_display = image.permute(1, 2, 0).numpy()
            img_display = img_display * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
            img_display = np.clip(img_display, 0, 1)

            #2
            gt_mask = mask.numpy()

            #plot
            axes[i, 0].imshow(img_display)
            axes[i, 0].set_title(f"Original Image")
            axes[i, 0].axis('off')

            axes[i, 1].imshow(gt_mask, cmap='tab10', vmin=0, vmax=7)
            axes[i, 1].set_title(f"Ground Truth")
            axes[i, 1].axis('off')

            axes[i, 2].imshow(prediction, cmap='tab10', vmin=0, vmax=7)
            axes[i, 2].set_title(f"Model Prediction")
            axes[i, 2].axis('off')

    plt.tight_layout()
    plt.show()

visualize_predictions(model, train_dataset, device, num_samples=4)